In [2]:
# --- [1] Load & Preprocess Data ---
import pandas as pd
import numpy as np

df = pd.read_csv("code/household_power_consumption.txt", sep=";", low_memory=False, na_values="?")
df.columns = df.columns.str.strip()
df["DateTime"] = pd.to_datetime(df["Date"] + ' ' + df["Time"], format="%d/%m/%Y %H:%M:%S", errors='coerce')
df.drop(columns=["Date", "Time"], inplace=True)

# Convert numeric columns
cols_to_convert = df.columns.difference(["DateTime"])
for col in cols_to_convert:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df.dropna(inplace=True)

# Define features and target
X = df[['Global_reactive_power', 'Voltage', 'Global_intensity', 
        'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']]
y = df['Global_active_power']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

print("🔹 Linear Regression:")
print("MAE:", mean_absolute_error(y_test, y_pred_lr))
print("MSE:", mean_squared_error(y_test, y_pred_lr))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_lr)))
print("R²:", r2_score(y_test, y_pred_lr))


🔹 Linear Regression:
MAE: 0.025838930356817672
MSE: 0.0016296157203171905
RMSE: 0.040368499109047766
R²: 0.9985500910441268


In [7]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

rf = RandomForestRegressor(random_state=42)
param_grid_rf = {
    'n_estimators': [2, 5],
    'max_depth': [1, 5],
    'min_samples_leaf': [1, 2]
}

grid_rf = GridSearchCV(rf, param_grid_rf , cv=3, scoring='r2', n_jobs=-1)
grid_rf.fit(X_train, y_train)
y_pred_rf = grid_rf.predict(X_test)
   
print("\n🌲 Random Forest (Tuned):")
print("Best Params:", grid_rf.best_params_)
print("MAE:", mean_absolute_error(y_test, y_pred_rf))
print("MSE:", mean_squared_error(y_test, y_pred_rf))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_rf)))
print("R²:", r2_score(y_test, y_pred_rf))



🌲 Random Forest (Tuned):
Best Params: {'max_depth': 5, 'min_samples_leaf': 1, 'n_estimators': 5}
MAE: 0.03710969529351562
MSE: 0.003367674378113894
RMSE: 0.05803166702856204
R²: 0.9970036977549889


In [8]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

gbr = GradientBoostingRegressor(random_state=42)
param_grid_gbr = {
    'n_estimators': [5],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5]
}

grid_gbr = GridSearchCV(gbr, param_grid_gbr, cv=3, scoring='r2', n_jobs=-1)
grid_gbr.fit(X_train, y_train)
y_pred_gbr = grid_gbr.predict(X_test)

print("\n🚀 Gradient Boosting (Tuned):")
print("Best Params:", grid_gbr.best_params_)
print("MAE:", mean_absolute_error(y_test, y_pred_gbr))
print("MSE:", mean_squared_error(y_test, y_pred_gbr))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_gbr)))
print("R²:", r2_score(y_test, y_pred_gbr))



🚀 Gradient Boosting (Tuned):
Best Params: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 5}
MAE: 0.4847592396561147
MSE: 0.39422279317846903
RMSE: 0.6278716375012244
R²: 0.6492503408548754


In [4]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert to tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

# DataLoader
train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=128, shuffle=True)

# Define small neural net
class SimpleNet(nn.Module):
    def __init__(self):
        super(SimpleNet, self).__init__()
        self.fc1 = nn.Linear(X_train_tensor.shape[1], 16)
        self.out = nn.Linear(16, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        return self.out(x)

model = SimpleNet()

# Training setup
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Fast training (5 epochs)
for epoch in range(5):
    for inputs, targets in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch} - Loss: {loss.item():.4f}")

# Evaluation
model.eval()
with torch.no_grad():
    predictions = model(X_test_tensor).numpy()

print("\n🧠 PyTorch Neural Network (Lightweight):")
print("MAE:", mean_absolute_error(y_test, predictions))
print("MSE:", mean_squared_error(y_test, predictions))
print("RMSE:", np.sqrt(mean_squared_error(y_test, predictions)))
print("R²:", r2_score(y_test, predictions))


Epoch 0 - Loss: 0.0007
Epoch 1 - Loss: 0.0008
Epoch 2 - Loss: 0.0009
Epoch 3 - Loss: 0.0020
Epoch 4 - Loss: 0.0008

🧠 PyTorch Neural Network (Lightweight):
MAE: 0.020871093350827492
MSE: 0.0011571899383698843
RMSE: 0.03401749459278099
R²: 0.9989704198147019
